# Frozen CAMeLBERT representations: likely escalation

Train a separate one-logit linear head using a frozen copy of the saved five-intent CAMeLBERT encoder. Original intent artifacts are read-only. Target `escalation=1` means **likely escalation**: explicit escalation or strong unresolved-service indicators. Labels are not changed. No team-call data is used.

**Evaluation limitation:** the encoder previously learned intent labels from all training records, and its checkpoint was selected using intent development loss. Grouped CV below isolates escalation-head fitting and selection, but is not an unbiased end-to-end CV estimate of encoder training. The existing development set is already development data, not an untouched test set. No escalation development labels enter head/threshold selection.

Run sequentially in **Python (CallSense)**. MPS is required; unavailable MPS or CPU-fallback settings stop execution. Lightweight preflight may execute imports, data checks, and fold construction only. Feature extraction, head fitting, development inference, and export must wait for authorization.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlparse
import hashlib
import json
import re
import sys

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, classification_report,
                             confusion_matrix, f1_score, precision_score, recall_score)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "notebooks/08_camelbert_escalation.ipynb").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from inside the CallSense project")
TRAIN_PATH = ROOT / "data/train_reviewed.csv"
DEV_PATH = ROOT / "data/processed/val.csv"
SEED = 42
N_SPLITS = 3
C_VALUES = [0.1, 1.0, 10.0]
THRESHOLDS = [round(x / 100, 2) for x in range(10, 91, 5)]

import random
import torch
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import os

ENCODER_DIR = ROOT / "models/camelbert_intent_reviewed_20261030T150108_941481Z"
BASELINE_DIR = ROOT / "models/escalation_tfidf_20261030T182407_857845Z"
BASELINE_EVAL = ROOT / "eval/escalation_tfidf_20261030T182407_857845Z"
MAX_LENGTH = 128
FEATURE_BATCH_SIZE = 8
HEAD_CANDIDATES = [
    {"learning_rate": lr, "epochs": epochs}
    for lr in [0.001, 0.01] for epochs in [20, 50]
]


## Data and provenance checks

Preserve CSV row order. Stop on invalid labels or cross-split ID, author, post or normalized-text overlap. Within training, connected author/post/text groups stay in the same fold. Original CSVs are never written.


In [ ]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def clean_text(text):
    # Same normalization as the existing keyword/TF-IDF baseline.
    text = text.lower()
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = re.sub("[أإآٱ]", "ا", text)
    return text.replace("ة", "ه").replace("ى", "ي").replace("ـ", "")

def author_from_url(value):
    url = urlparse(value if "://" in value else "https://" + value)
    parts = url.path.strip("/").split("/")
    host = url.netloc.lower().removeprefix("www.")
    if host in {"x.com", "twitter.com", "mobile.twitter.com", "mobile.x.com"} and len(parts) >= 3 and parts[1] == "status":
        return parts[0].lower()
    raise ValueError(f"Cannot resolve author from source URL: {value}")

def canonical_post(value):
    url = urlparse(value if "://" in value else "https://" + value)
    match = re.search(r"/status/(\d+)", url.path)
    return match.group(1) if match else url.netloc.lower() + url.path.rstrip("/")

input_hashes = {str(p.relative_to(ROOT)): sha256(p) for p in [TRAIN_PATH, DEV_PATH]}
train = pd.read_csv(TRAIN_PATH)
dev = pd.read_csv(DEV_PATH)
for name, frame in [("training", train), ("development", dev)]:
    required = {"complaint_id", "model_text", "source_url", "escalation"}
    if frame.empty or not required <= set(frame.columns):
        raise ValueError(f"{name}: empty input or missing required columns")
    for col in ["complaint_id", "model_text", "source_url"]:
        if frame[col].isna().any() or not frame[col].map(lambda x: isinstance(x, str) and bool(x.strip())).all():
            raise ValueError(f"{name}: missing or invalid {col}")
    if frame["complaint_id"].duplicated().any():
        raise ValueError(f"{name}: duplicate complaint IDs")
    if not frame["escalation"].isin([0, 1]).all():
        raise ValueError(f"{name}: escalation must be binary and nonmissing")
    frame["author_group"] = frame["source_url"].map(author_from_url)
    frame["canonical_post"] = frame["source_url"].map(canonical_post)
    frame["normalized_text"] = frame["model_text"].map(clean_text).str.replace(r"\s+", " ", regex=True).str.strip()
    if frame["normalized_text"].eq("").any():
        raise ValueError(f"{name}: empty normalized text")
for col in ["complaint_id", "author_group", "canonical_post", "normalized_text"]:
    if set(train[col]) & set(dev[col]):
        raise ValueError(f"Training/development overlap in {col}")
print("Input validation passed; no split or label changes")
display(pd.DataFrame({"training": train.escalation.value_counts(),
                      "development": dev.escalation.value_counts()}).fillna(0).astype(int))

source_settings = json.loads((ENCODER_DIR / "settings.json").read_text())
baseline_settings = json.loads((BASELINE_DIR / "settings.json").read_text())
# Do not load baseline development metrics until escalation selection is complete.
for relative, digest in input_hashes.items():
    if source_settings["input_sha256"].get(relative) != digest:
        raise ValueError("Encoder data provenance mismatch: " + relative)
    if baseline_settings["input_sha256"].get(relative) != digest:
        raise ValueError("Baseline data provenance mismatch: " + relative)
if not (BASELINE_EVAL / "metrics.json").is_file():
    raise FileNotFoundError("Saved TF-IDF metrics are required for comparison")
if source_settings["max_length"] != MAX_LENGTH:
    raise ValueError("Encoder preprocessing max_length mismatch")
source_hashes = {str(p.relative_to(ROOT)): sha256(p)
                 for p in ENCODER_DIR.iterdir() if p.is_file()}


## Training-only groups

Three escalation-stratified folds, grouping connected authors, posts and normalized duplicate texts. Both classes must exist on each side. Normalized text is used for leakage checks only; encoder input remains the original `model_text` with its saved tokenizer.


In [ ]:
y_train = train.escalation.to_numpy(dtype=int)
X_train = train.model_text.map(clean_text)
# Connected groups keep shared authors, posts, and normalized duplicates together.
parent = list(range(len(train)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i
for column in ["author_group", "canonical_post", "normalized_text"]:
    first_seen = {}
    for i, value in enumerate(train[column]):
        if value in first_seen:
            parent[find(i)] = find(first_seen[value])
        else:
            first_seen[value] = i
groups = np.array([find(i) for i in range(len(train))])
cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
splits = list(cv.split(X_train, y_train, groups))
fold_summary = []
for fold, (fit_idx, hold_idx) in enumerate(splits, 1):
    if set(groups[fit_idx]) & set(groups[hold_idx]):
        raise ValueError("Author overlap inside CV")
    if set(y_train[fit_idx]) != {0, 1} or set(y_train[hold_idx]) != {0, 1}:
        raise ValueError("Each CV partition must contain both escalation classes")
    fold_summary.append({"fold": fold, "fit_records": len(fit_idx),
                         "fit_positives": int(y_train[fit_idx].sum()),
                         "heldout_records": len(hold_idx),
                         "heldout_positives": int(y_train[hold_idx].sum())})
display(pd.DataFrame(fold_summary))


## Frozen encoder initialization — no training

Load only local saved files. Freeze every encoder parameter and use `eval()` to disable dropout. The intent head is not used. MPS is strictly required. No encoder forward pass occurs in this cell.


In [ ]:
if not torch.backends.mps.is_available():
    raise RuntimeError("MPS required; use Python (CallSense) in VS Code")
if os.environ.get("PYTORCH_ENABLE_MPS_FALLBACK") == "1":
    raise RuntimeError("Disable MPS CPU fallback and restart the kernel")
device = torch.device("mps")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(ENCODER_DIR, local_files_only=True)
intent_model = AutoModelForSequenceClassification.from_pretrained(ENCODER_DIR, local_files_only=True)
encoder = intent_model.base_model
for parameter in encoder.parameters():
    parameter.requires_grad_(False)
encoder = encoder.to(device).eval()
del intent_model
assert not any(p.requires_grad for p in encoder.parameters())
HIDDEN_SIZE = encoder.config.hidden_size
print("Frozen encoder:", HIDDEN_SIZE, "features; device:", device)


## Training features and head-only selection

Extract pooled BERT representations with no gradients. No development features or labels are used for selection. Each fold's `pos_weight = negatives / positives` uses only that fold's fitting labels. Fit fresh `Linear(hidden_size, 1)` heads using BCEWithLogitsLoss and AdamW (weight decay 0.01), full-batch updates, with no encoder updates.

Choose learning rate, epoch count and threshold by pooled OOF positive F1. Tie-break: candidate order shown above, then threshold closest to 0.50, then lower threshold. These are tuning scores. After selection, fit a fresh head on all training features, using all-training imbalance statistics and the selected fixed epoch count.


In [ ]:
def extract_features(frame):
    encoder.eval()
    chunks = []
    with torch.no_grad():
        for start in range(0, len(frame), FEATURE_BATCH_SIZE):
            inputs = tokenizer(frame.iloc[start:start + FEATURE_BATCH_SIZE].model_text.tolist(),
                               padding="max_length", truncation=True,
                               max_length=MAX_LENGTH, return_tensors="pt")
            output = encoder(input_ids=inputs["input_ids"].to(device),
                             attention_mask=inputs["attention_mask"].to(device))
            features = output.pooler_output
            if not torch.isfinite(features).all().item():
                raise RuntimeError("Non-finite encoder features")
            chunks.append(features.cpu())
    return torch.cat(chunks)

def fit_head(features, labels, configuration):
    torch.manual_seed(SEED)
    head = torch.nn.Linear(HIDDEN_SIZE, 1).to(device)
    positives = int(labels.sum())
    negatives = len(labels) - positives
    if min(positives, negatives) == 0:
        raise ValueError("Both classes required for head fitting")
    pos_weight = negatives / positives
    criterion = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))
    optimizer = torch.optim.AdamW(head.parameters(), lr=configuration["learning_rate"], weight_decay=0.01)
    X = features.to(device)
    y = torch.tensor(labels, dtype=torch.float32, device=device).reshape(-1, 1)
    history = []
    head.train()
    for epoch in range(1, configuration["epochs"] + 1):
        optimizer.zero_grad()
        loss = criterion(head(X), y)
        if not torch.isfinite(loss).item():
            raise RuntimeError("Non-finite head loss")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0, error_if_nonfinite=True)
        optimizer.step()
        history.append({"epoch": epoch, "loss": loss.item()})
    return head.eval(), history, pos_weight

def predict_head(head, features):
    head.eval()
    with torch.no_grad():
        probabilities = torch.sigmoid(head(features.to(device))).squeeze(1).cpu().numpy()
    if not np.isfinite(probabilities).all():
        raise RuntimeError("Non-finite head probabilities")
    return probabilities

# Feature extraction and training start here; do not run during preflight.
train_features = extract_features(train)
search_rows, oof_by_candidate = [], {}
for candidate_id, configuration in enumerate(HEAD_CANDIDATES):
    oof = np.full(len(train), np.nan)
    for fit_idx, hold_idx in splits:
        head, _, _ = fit_head(train_features[fit_idx], y_train[fit_idx], configuration)
        oof[hold_idx] = predict_head(head, train_features[hold_idx])
    if not np.isfinite(oof).all():
        raise RuntimeError("Incomplete OOF predictions")
    oof_by_candidate[candidate_id] = oof
    for threshold in THRESHOLDS:
        predictions = (oof >= threshold).astype(int)
        search_rows.append({"candidate_id": candidate_id, **configuration, "threshold": threshold,
            "positive_f1": float(f1_score(y_train, predictions, zero_division=0)),
            "fold_f1": [float(f1_score(y_train[idx], predictions[idx], zero_division=0)) for _, idx in splits]})
selected = sorted(search_rows, key=lambda r: (-r["positive_f1"], r["candidate_id"],
                                             abs(r["threshold"] - 0.5), r["threshold"]))[0]
selected_threshold = selected["threshold"]
selected_configuration = HEAD_CANDIDATES[selected["candidate_id"]]
final_head, final_history, final_pos_weight = fit_head(train_features, y_train, selected_configuration)
print("Training-only selection:", selected)


## One development evaluation and saved TF-IDF comparison

Use the fixed head and threshold once. PR-AUC is average precision. Do not change settings based on these results. Confusion matrices use rows=actual, columns=predicted, ordered [0, 1]. Saved TF-IDF results are identified as prior results, not recomputed outputs.


In [ ]:
def evaluate(y_true, scores, threshold):
    predictions = (scores >= threshold).astype(int)
    return predictions, {
        "positive_precision": float(precision_score(y_true, predictions, zero_division=0)),
        "positive_recall": float(recall_score(y_true, predictions, zero_division=0)),
        "positive_f1": float(f1_score(y_true, predictions, zero_division=0)),
        "macro_f1": float(f1_score(y_true, predictions, average="macro", zero_division=0)),
        "pr_auc_average_precision": float(average_precision_score(y_true, scores)),
        "confusion_matrix_labels": [0, 1],
        "confusion_matrix": confusion_matrix(y_true, predictions, labels=[0, 1]).tolist(),
        "positive_support": int(y_true.sum()), "records": len(y_true),
        "classification_report": classification_report(y_true, predictions, labels=[0, 1],
            target_names=["no_likely_escalation", "likely_escalation"],
            output_dict=True, zero_division=0),
    }

y_dev = dev.escalation.to_numpy(dtype=int)
dev_features = extract_features(dev)
dev_probabilities = predict_head(final_head, dev_features)
dev_predictions, neural_metrics = evaluate(y_dev, dev_probabilities, selected_threshold)
saved_baseline = json.loads((BASELINE_EVAL / "metrics.json").read_text())["tfidf_logistic"]
metrics = {"frozen_camelbert_head": neural_metrics, "saved_tfidf_baseline": saved_baseline}
comparison_columns = ["positive_precision", "positive_recall", "positive_f1", "pr_auc_average_precision", "macro_f1"]
display(pd.DataFrame({name: {k: values[k] for k in comparison_columns}
                      for name, values in metrics.items()}).T)
print("Frozen head confusion matrix:", neural_metrics["confusion_matrix"])
print("Saved TF-IDF confusion matrix:", saved_baseline["confusion_matrix"])
predictions = dev[["complaint_id", "model_text", "escalation"]].copy()
predictions["escalation_probability"] = dev_probabilities
predictions["escalation_prediction"] = dev_predictions
predictions["error_type"] = np.where((y_dev == 0) & (dev_predictions == 1), "false_positive",
                                      np.where((y_dev == 1) & (dev_predictions == 0), "false_negative", ""))
errors = predictions.loc[predictions.error_type.ne("")].copy()
display(errors)


## New versioned export

Save an independent frozen encoder/tokenizer copy and a CPU state dictionary for the one-logit linear head. To reload, use AutoModel on `encoder/`, then Linear(hidden_size, 1), load `head.pt`, set both modules to eval(), apply pooler_output, sigmoid and the saved threshold. Keep original model artifacts untouched. Export fails if input hashes change or an output directory already exists.


In [ ]:
for relative, digest in {**input_hashes, **source_hashes}.items():
    if sha256(ROOT / relative) != digest:
        raise RuntimeError("Input artifact changed during run: " + relative)
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
model_dir = ROOT / "models" / f"camelbert_escalation_frozen_{run_id}"
eval_dir = ROOT / "eval" / f"camelbert_escalation_frozen_{run_id}"
model_dir.mkdir(parents=True, exist_ok=False)
eval_dir.mkdir(parents=True, exist_ok=False)
encoder.save_pretrained(model_dir / "encoder", safe_serialization=True)
tokenizer.save_pretrained(model_dir / "encoder")
torch.save({name: value.detach().cpu().clone() for name, value in final_head.state_dict().items()},
           model_dir / "head.pt")
metadata = {
    "target": "escalation", "definition": "likely escalation: explicit escalation or strong unresolved-service indicators",
    "encoder_source": str(ENCODER_DIR), "encoder_source_sha256": source_hashes,
    "input_sha256": input_hashes, "baseline_source": str(BASELINE_EVAL),
    "encoder_frozen": True, "pooling": "pooler_output", "head": "Linear(hidden_size, 1)",
    "hidden_size": HIDDEN_SIZE, "loss": "BCEWithLogitsLoss", "pos_weight": final_pos_weight,
    "imbalance_statistics": "fold fitting partitions only; full training for final fit",
    "seed": SEED, "device": str(device), "max_length": MAX_LENGTH,
    "feature_batch_size": FEATURE_BATCH_SIZE, "head_batch": "full training partition",
    "optimizer": "AdamW", "weight_decay": 0.01, "gradient_clip_norm": 1.0,
    "candidates": HEAD_CANDIDATES, "threshold_candidates": THRESHOLDS,
    "selected_configuration": selected_configuration, "selected_oof_result": selected,
    "threshold": selected_threshold, "probability_function": "sigmoid", "threshold_comparison": ">=",
    "fold_summary": fold_summary, "grouping": "connected authors/posts/normalized text",
    "selection_metric": "training pooled OOF positive F1",
    "tie_break": "candidate order, threshold closest to .50, lower threshold",
    "evaluation_limit": "head-only CV with previously intent-supervised encoder; development is not untouched test data",
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                 "sklearn": sklearn.__version__, "python": sys.version},
    "final_training_history": final_history,
}
for path, content in [(model_dir / "settings.json", metadata), (eval_dir / "metrics.json", metrics)]:
    with path.open("x", encoding="utf-8") as handle:
        json.dump(content, handle, ensure_ascii=False, indent=2, allow_nan=False)
predictions.to_csv(eval_dir / "predictions.csv", index=False, mode="x")
errors.to_csv(eval_dir / "errors.csv", index=False, mode="x")
pd.DataFrame(search_rows).to_csv(eval_dir / "cv_search.csv", index=False, mode="x")
oof = train[["complaint_id", "escalation", "author_group"]].copy()
oof["cv_group"] = groups
oof["probability"] = oof_by_candidate[selected["candidate_id"]]
oof["prediction"] = (oof.probability >= selected_threshold).astype(int)
oof.to_csv(eval_dir / "training_oof_predictions.csv", index=False, mode="x")
print("Model:", model_dir)
print("Evaluation:", eval_dir)
